In [6]:
data_dir = '/scratch/mjehangir/tcga/'

In [7]:
setwd(data_dir)

In [14]:
library(TCGAbiolinks)
library(tidyverse)
library(maftools)
library(SummarizedExperiment)
library(ggplot2)
library(ComplexHeatmap)
library(circlize)
library(dplyr)
library(tidyr)
library(ggforce) 
library(tidyverse)
library(BSgenome.Hsapiens.UCSC.hg38)   # for seqlengths()
# For geom_sina()

In [9]:
glioma_idh_status = read.csv(file = "/home/mjehangir/glioma_manuscript/tcga_analysis/glioma_status.txt", header = TRUE, sep = "\t")
head(glioma_idh_status)

,WT,Non.codel,codel
,<chr>,<chr>,<chr>
1,TCGA-CS-4941,TCGA-CS-4938,TCGA-CS-5390
2,TCGA-CS-5395,TCGA-CS-4942,TCGA-CS-5396
3,TCGA-CS-5397,TCGA-CS-4943,TCGA-CS-6668
4,TCGA-CS-6186,TCGA-CS-4944,TCGA-CS-6670
5,TCGA-CS-6188,TCGA-CS-5393,TCGA-DB-5274
6,TCGA-CS-6669,TCGA-CS-5394,TCGA-DB-5278


In [10]:
CNV_data <- read.delim("merged_CNV_stats.txt", header = TRUE)


In [17]:
head(CNV_data)
head(glioma_idh_status)

,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>
1,TCGA-02-0001-01C-01D-0182-01,chr18,18p,0,1,0.9635290,0
2,TCGA-02-0001-10A-01D-0182-01,chr18,18p,0,1,1.0011097,0
3,TCGA-02-0003-01A-01D-0182-01,chr18,18p,0,1,1.0475361,0
4,TCGA-02-0003-10A-01D-0182-01,chr18,18p,0,1,1.0009015,0
5,TCGA-02-0006-01B-01D-0182-01,chr18,18p,-1,1,0.7738361,0
6,TCGA-02-0006-10A-01D-0182-01,chr18,18p,0,1,1.0009709,0


,WT,Non.codel,codel
,<chr>,<chr>,<chr>
1,TCGA-CS-4941,TCGA-CS-4938,TCGA-CS-5390
2,TCGA-CS-5395,TCGA-CS-4942,TCGA-CS-5396
3,TCGA-CS-5397,TCGA-CS-4943,TCGA-CS-6668
4,TCGA-CS-6186,TCGA-CS-4944,TCGA-CS-6670
5,TCGA-CS-6188,TCGA-CS-5393,TCGA-DB-5274
6,TCGA-CS-6669,TCGA-CS-5394,TCGA-DB-5278


In [ ]:
nrow(CNV_data)
nrow(ptel)
nrow(glioma_idh_status)

In [ ]:
unique(ptel$disease)

In [16]:
ptel = read.delim("ptel_sample_disease.tsv", sep = "\t", header = TRUE)

In [18]:
head(ptel)

,patient_id,disease
,<chr>,<chr>
1,TCGA-02-0003,GBM
2,TCGA-02-0033,GBM
3,TCGA-02-0047,GBM
4,TCGA-02-2483,GBM
5,TCGA-02-2485,GBM
6,TCGA-02-2486,GBM


In [23]:
# Load necessary libraries
library(dplyr)
library(tidyr)

# 1. Assume your three data frames are named as follows:
#    CNV_data           : contains columns SampleID, seqnames, arm_call, etc.
#    ptel               : contains columns patient_id and disease
#    glioma_idh_status  : contains columns WT, Non.codel, and codel, each listing patient IDs

# 2. Extract patient_id from CNV_data$SampleID
#    Here we assume that the patient identifier is the first 12 characters of SampleID,
#    e.g. from "TCGA-02-0001-01C-01D-0182-01" we extract "TCGA-02-0001"

CNV_data2 <- CNV_data %>%
  mutate(
    patient_id = substr(SampleID, 1, 12)
  )

In [24]:
head(CNV_data2)

,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd,patient_id
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>
1,TCGA-02-0001-01C-01D-0182-01,chr18,18p,0,1,0.9635290,0,TCGA-02-0001
2,TCGA-02-0001-10A-01D-0182-01,chr18,18p,0,1,1.0011097,0,TCGA-02-0001
3,TCGA-02-0003-01A-01D-0182-01,chr18,18p,0,1,1.0475361,0,TCGA-02-0003
4,TCGA-02-0003-10A-01D-0182-01,chr18,18p,0,1,1.0009015,0,TCGA-02-0003
5,TCGA-02-0006-01B-01D-0182-01,chr18,18p,-1,1,0.7738361,0,TCGA-02-0006
6,TCGA-02-0006-10A-01D-0182-01,chr18,18p,0,1,1.0009709,0,TCGA-02-0006


In [25]:
merged1 <- CNV_data2 %>%
  left_join(ptel, by = "patient_id")

In [26]:
head(merged1)

,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd,patient_id,disease
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<chr>
1,TCGA-02-0001-01C-01D-0182-01,chr18,18p,0,1,0.9635290,0,TCGA-02-0001,NA
2,TCGA-02-0001-10A-01D-0182-01,chr18,18p,0,1,1.0011097,0,TCGA-02-0001,NA
3,TCGA-02-0003-01A-01D-0182-01,chr18,18p,0,1,1.0475361,0,TCGA-02-0003,GBM
4,TCGA-02-0003-10A-01D-0182-01,chr18,18p,0,1,1.0009015,0,TCGA-02-0003,GBM
5,TCGA-02-0006-01B-01D-0182-01,chr18,18p,-1,1,0.7738361,0,TCGA-02-0006,NA
6,TCGA-02-0006-10A-01D-0182-01,chr18,18p,0,1,1.0009709,0,TCGA-02-0006,NA


In [27]:
# 4. Pivot glioma_idh_status into long form
glioma_subtypes_long <- glioma_idh_status %>%
  pivot_longer(
    cols      = c(WT, `Non.codel`, codel),
    names_to  = "subtype",
    values_to = "patient_id"
  ) %>%
  filter(!is.na(patient_id))

In [28]:
head(glioma_subtypes_long)

subtype,patient_id
<chr>,<chr>
WT,TCGA-CS-4941
Non.codel,TCGA-CS-4938
codel,TCGA-CS-5390
WT,TCGA-CS-5395
Non.codel,TCGA-CS-4942
codel,TCGA-CS-5396


In [29]:
# 5. Join merged1 with glioma_subtypes_long
merged2 <- merged1 %>%
  left_join(glioma_subtypes_long, by = "patient_id")

# 6. Select only the columns you need
final_df <- merged2 %>%
  select(
    SampleID,
    seqnames,
    arm,
    arm_call,
    disease,
    subtype
  )

In [30]:
head(merged2)

,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd,patient_id,disease,subtype
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,TCGA-02-0001-01C-01D-0182-01,chr18,18p,0,1,0.9635290,0,TCGA-02-0001,NA,WT
2,TCGA-02-0001-10A-01D-0182-01,chr18,18p,0,1,1.0011097,0,TCGA-02-0001,NA,WT
3,TCGA-02-0003-01A-01D-0182-01,chr18,18p,0,1,1.0475361,0,TCGA-02-0003,GBM,WT
4,TCGA-02-0003-10A-01D-0182-01,chr18,18p,0,1,1.0009015,0,TCGA-02-0003,GBM,WT
5,TCGA-02-0006-01B-01D-0182-01,chr18,18p,-1,1,0.7738361,0,TCGA-02-0006,NA,WT
6,TCGA-02-0006-10A-01D-0182-01,chr18,18p,0,1,1.0009709,0,TCGA-02-0006,NA,WT


In [31]:
# Filter merged2 for rows where subtype is "WT" and disease is NA
wt_na_rows <- merged2 %>%
  filter(subtype == "codel" & is.na(disease))

# Print those rows (you can use head() if you only want to preview)
nrow(wt_na_rows)
# Or, to see just the first few:
head(wt_na_rows)


[1] 390

,SampleID,seqnames,arm,arm_call,arm_num_seg,arm_cr_wmean,arm_cr_wsd,patient_id,disease,subtype
,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,TCGA-02-0028-01A-01D-0182-01,chr18,18p,0,1,1.005700,0,TCGA-02-0028,NA,codel
2,TCGA-02-0028-01A-01D-0182-01,chr17,17p,0,1,1.037851,0,TCGA-02-0028,NA,codel
3,TCGA-02-0028-10A-01D-0182-01,chr17,17p,0,1,1.001179,0,TCGA-02-0028,NA,codel
4,TCGA-02-0028-01A-01D-0182-01,chr10,10p,0,1,1.013608,0,TCGA-02-0028,NA,codel
5,TCGA-02-0028-01A-01D-0182-01,chr12,12p,0,1,1.012344,0,TCGA-02-0028,NA,codel
6,TCGA-02-0028-10A-01D-0182-01,chr10,10p,0,1,1.002776,0,TCGA-02-0028,NA,codel


In [32]:
unique(wt_na_rows$patient_id)

[1] "TCGA-02-0028" "TCGA-08-0344" "TCGA-CS-5390" "TCGA-HT-A619" "TCGA-R8-A6ML"
[6] "TCGA-R8-A73M"

In [33]:
unique(merged2$subtype)

[1] "WT"        "Non.codel" "codel"     NA

In [34]:
nrow(merged2)

[1] 693864

In [35]:
# 7. Inspect
head(final_df)

,SampleID,seqnames,arm,arm_call,disease,subtype
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>
1,TCGA-02-0001-01C-01D-0182-01,chr18,18p,0,NA,WT
2,TCGA-02-0001-10A-01D-0182-01,chr18,18p,0,NA,WT
3,TCGA-02-0003-01A-01D-0182-01,chr18,18p,0,GBM,WT
4,TCGA-02-0003-10A-01D-0182-01,chr18,18p,0,GBM,WT
5,TCGA-02-0006-01B-01D-0182-01,chr18,18p,-1,NA,WT
6,TCGA-02-0006-10A-01D-0182-01,chr18,18p,0,NA,WT


In [36]:
colnames(final_df)


[1] "SampleID" "seqnames" "arm"      "arm_call" "disease"  "subtype"

In [45]:
# Convert SampleID to character

tss_lookup <- data.frame(
  TSS = c("01", "02", "03", "04", "05", "06", "07", "08", "09", "10",
          "11", "12", "13", "14", "15", "16", "17", "18", "19", "20",
          "21", "22", "23", "24", "25", "26", "27", "28", "29", "30",
          "31", "32"),
  Disease = c("BRCA", "GBM", "OV", "LUAD", "KIRC", "COAD", "READ", "LUSC", "PRAD", "THCA",
              "BLCA", "UCEC", "HNSC", "LIHC", "STAD", "SKCM", "ESCA", "CESC", "PCPG", "TGCT",
              "THYM", "CHOL", "ACC", "MESO", "UCS", "DLBC", "SARC", "KIRP", "LGG", "KICH",
              "LAML", "PAAD")
)
final_df$SampleID <- as.character(final_df$SampleID)

# Extract TSS from SampleID
final_df$TSS <- sapply(strsplit(final_df$SampleID, "-"), function(x) x[2])

# Replace missing or "NA" disease entries based on TSS lookup
final_df$disease[is.na(final_df$disease) | final_df$disease == "NA"] <- 
  tss_lookup$Disease[match(final_df$TSS[is.na(final_df$disease) | final_df$disease == "NA"], 
                           tss_lookup$TSS)]

# Remove the helper TSS column
final_df$TSS <- NULL

head(final_df)

,SampleID,seqnames,arm,arm_call,disease,subtype
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>
1,TCGA-02-0001-01C-01D-0182-01,chr18,18p,0,GBM,WT
2,TCGA-02-0001-10A-01D-0182-01,chr18,18p,0,GBM,WT
3,TCGA-02-0003-01A-01D-0182-01,chr18,18p,0,GBM,WT
4,TCGA-02-0003-10A-01D-0182-01,chr18,18p,0,GBM,WT
5,TCGA-02-0006-01B-01D-0182-01,chr18,18p,-1,GBM,WT
6,TCGA-02-0006-10A-01D-0182-01,chr18,18p,0,GBM,WT


In [46]:
final_df <- final_df %>%
  dplyr::rename(
    chromosome = seqnames,
    status = arm_call
  )
head(final_df)

,SampleID,chromosome,arm,status,disease,subtype
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>
1,TCGA-02-0001-01C-01D-0182-01,chr18,18p,0,GBM,WT
2,TCGA-02-0001-10A-01D-0182-01,chr18,18p,0,GBM,WT
3,TCGA-02-0003-01A-01D-0182-01,chr18,18p,0,GBM,WT
4,TCGA-02-0003-10A-01D-0182-01,chr18,18p,0,GBM,WT
5,TCGA-02-0006-01B-01D-0182-01,chr18,18p,-1,GBM,WT
6,TCGA-02-0006-10A-01D-0182-01,chr18,18p,0,GBM,WT


In [55]:
write.table(final_df, "IDH_and_1p19q_subtypes_per_sample_v2.tsv", 
            sep = "\t", row.names = FALSE, quote = FALSE)


In [41]:
# Check how many rows have disease == NA or "NA"
sum(is.na(final_df$disease) | final_df$disease == "NA")


[1] 86826

In [39]:
nrow(final_df)

[1] 693864

In [53]:
wt_na_rows <- final_df %>%
  filter(subtype == "WT" & is.na(disease))
head(wt_na_rows)
nrow(wt_na_rows)

,SampleID,chromosome,arm,status,disease,subtype
,<chr>,<chr>,<chr>,<int>,<chr>,<chr>
1,TCGA-P5-A5EY-01A-11D-A27J-01,chr18,18p,0,NA,WT
2,TCGA-P5-A5EY-10A-01D-A27M-01,chr18,18p,0,NA,WT
3,TCGA-P5-A5EY-01A-11D-A27J-01,chr5,5p,0,NA,WT
4,TCGA-P5-A5EY-10A-01D-A27M-01,chr5,5p,0,NA,WT
5,TCGA-P5-A5EY-01A-11D-A27J-01,chr4,4p,0,NA,WT
6,TCGA-P5-A5EY-10A-01D-A27M-01,chr4,4p,0,NA,WT


[1] 177

In [54]:
unique_ids <- unique(wt_na_rows$SampleID)
print(unique_ids)


[1] "TCGA-P5-A5EY-01A-11D-A27J-01" "TCGA-P5-A5EY-10A-01D-A27M-01"
[3] "TCGA-TM-A84B-12A-01D-A366-01" "TCGA-TM-A84C-01A-11D-A36N-01"
[5] "TCGA-TM-A84C-12A-01D-A366-01" "TCGA-TM-A84B-01A-11D-A36N-01"


In [13]:
library(dplyr)
library(tidyr)
library(stringr)

# 1. Convert glioma subtype table to long format
glioma_subtypes <- glioma_idh_status %>%
  pivot_longer(cols = everything(), names_to = "subtype", values_to = "sample_id") %>%
  filter(!is.na(sample_id)) %>%
  mutate(subtype = recode(subtype,
                          WT = "IDH-wt",
                          Non.codel = "IDH-noncodel",
                          codel = "IDH-codel"))

# 2. Get chromosome-level status: -1 (deletion), +1 (amplification), 0 (neutral)
CNV_summary <- CNV_data %>%
  mutate(chromosome = seqnames) %>%
  group_by(SampleID, chromosome) %>%
  summarise(
    status = case_when(
      any(arm_call == -1) ~ -1,
      any(arm_call == 1)  ~ 1,
      TRUE                ~ 0
    ),
    .groups = "drop"
  )

# 3. Extract short TCGA ID for matching
CNV_summary <- CNV_summary %>%
  mutate(sample_id = str_extract(SampleID, "^TCGA-\\w{2}-\\w{4}"))

# 4. Join subtype info
final_table <- CNV_summary %>%
  left_join(glioma_subtypes, by = "sample_id") %>%
  mutate(
    subtype = if_else(is.na(subtype),
                      str_extract(SampleID, "^TCGA-\\w{2}") %>%
                        recode(
                          "TCGA-02" = "GBM",
                          "TCGA-06" = "LGG",
                          "TCGA-12" = "GBM",
                          "TCGA-19" = "GBM",
                          .default = "Other"),
                      subtype)
  ) %>%
  select(sample_id, subtype, chromosome, status) %>%
  distinct()

# Output preview
#print(head(final_table, 10))


In [14]:
#unique_sample_ids <- unique(final_table$sample_id)


In [27]:
ptel = read.delim("ptel_sample_disease.tsv", sep = "\t", header = TRUE)

In [28]:
head(ptel)

,patient_id,disease
,<chr>,<chr>
1,TCGA-02-0003,GBM
2,TCGA-02-0033,GBM
3,TCGA-02-0047,GBM
4,TCGA-02-2483,GBM
5,TCGA-02-2485,GBM
6,TCGA-02-2486,GBM


In [18]:
tablev1 = read.delim("IDH_and_1p19q_subtypes_per_sample.tsv", sep = "\t", header = TRUE)

In [20]:
head(tablev1)
head(ptel)

,sample_id,subtype,chromosome,status
,<chr>,<chr>,<chr>,<int>
1,TCGA-02-0001,IDH-wt,chr1,0
2,TCGA-02-0001,IDH-wt,chr10,0
3,TCGA-02-0001,IDH-wt,chr11,-1
4,TCGA-02-0001,IDH-wt,chr12,1
5,TCGA-02-0001,IDH-wt,chr13,0
6,TCGA-02-0001,IDH-wt,chr14,0


,sample_id,disease
,<chr>,<chr>
1,TCGA-02-0003-01,GBM
2,TCGA-02-0033-01,GBM
3,TCGA-02-0047-01,GBM
4,TCGA-02-2483-01,GBM
5,TCGA-02-2485-01,GBM
6,TCGA-02-2486-01,GBM


In [21]:
nrow(tablev1)
nrow(ptel)


[1] 309444

[1] 8953

In [29]:
library(dplyr)

# (1) Make sure your two data frames are in memory:
#     - tablev1:  columns = sample_id, subtype, chromosome, status
#     - ptel:     columns = patient_id, disease

# (2) Perform a left‐join, matching tablev1$sample_id → ptel$patient_id:
tablev1_with_disease <- 
  tablev1 %>%
  left_join(ptel, by = c("sample_id" = "patient_id"))

# (3) Verify dimensions and contents:
dim(tablev1_with_disease)   # should be 309444 rows × 5 columns
head(tablev1_with_disease)


[1] 309444      5

,sample_id,subtype,chromosome,status,disease
,<chr>,<chr>,<chr>,<int>,<chr>
1,TCGA-02-0001,IDH-wt,chr1,0,NA
2,TCGA-02-0001,IDH-wt,chr10,0,NA
3,TCGA-02-0001,IDH-wt,chr11,-1,NA
4,TCGA-02-0001,IDH-wt,chr12,1,NA
5,TCGA-02-0001,IDH-wt,chr13,0,NA
6,TCGA-02-0001,IDH-wt,chr14,0,NA


In [31]:
tail(tablev1_with_disease)


,sample_id,subtype,chromosome,status,disease
,<chr>,<chr>,<chr>,<int>,<chr>
309439,TCGA-ZX-AA5X,Other,chr6,0,NA
309440,TCGA-ZX-AA5X,Other,chr7,0,NA
309441,TCGA-ZX-AA5X,Other,chr8,0,NA
309442,TCGA-ZX-AA5X,Other,chr9,0,NA
309443,TCGA-ZX-AA5X,Other,chr1,0,NA
309444,TCGA-ZX-AA5X,Other,chr11,0,NA


In [16]:
write.table(final_table, "IDH_and_1p19q_subtypes_per_sample.tsv", 
            sep = "\t", row.names = FALSE, quote = FALSE)
